# VMBench workshop

Classify contest videos as **real (0)** or **generated (1)**, then explore motion diagnostics.

Components: **PAS** amplitude · **OIS** human structure · **TCS** coherence · **CAS** learned plausibility.

In [ ]:
from pathlib import Path
if not Path("/content/workshop/.git").exists():
    !git clone --recursive https://github.com/KK-one/vmbench-workshop.git /content/workshop
%cd /content/workshop

## 0. Initialization

Configure optional metrics and video selection. Set `RUN_METRICS=False` to only explore videos and prepare submissions.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "utils.py").is_file():
    PROJECT_DIR = Path("/content/workshop")
sys.path.insert(0, str(PROJECT_DIR))
from utils import (WorkshopConfig, configure, initialize, unpack_contest_archive, contest_video_paths,
                   browse_contest, write_binary_submission, compute_pas, compute_ois,
                   compute_tcs, compute_cas, compare_variants)

WORK_DIR = Path("/content")
DATASET_ZIP = WORK_DIR / "contest_vids.zip"  # ZIP uploaded to the Colab Files pane.
RUN_METRICS = True
METRICS_TO_RUN = ["PAS", "OIS", "TCS", "CAS"]
METRIC_VIDEO_IDS = []  # Choose HEX IDs after browsing; blank selects the first two.
MAX_METRIC_VIDEOS = 2
SUBJECT = "person"  # English object text for PAS/TCS; e.g. "dog" or "car".
CREATE_VARIANTS = False
workshop = configure(WorkshopConfig(work_dir=WORK_DIR, repo_dir=PROJECT_DIR / "vendor/vmbench",
                                   metrics_to_run=METRICS_TO_RUN))

## 1. Download and upload the dataset

[Join the competition](https://www.kaggle.com/t/ac5704b939234a8d96a93d8aabc66bde), accept its rules, and download the videos ZIP from **Data** (or use **Download All**).

In Colab, open **Files → Upload** and place the ZIP in `/content`. Wait for the upload to finish, then run the next cell. Nested ZIPs are supported; video bytes are preserved.


In [ ]:
if not DATASET_ZIP.is_file():
    archives = sorted(WORK_DIR.glob("*.zip"))
    if len(archives) != 1:
        raise FileNotFoundError("Upload the dataset ZIP to /content, or set DATASET_ZIP in cell 0.")
    DATASET_ZIP = archives[0]

CONTEST_DIR = WORK_DIR / "contest_vids"
CONTEST_VIDEO_PATHS = unpack_contest_archive(DATASET_ZIP, CONTEST_DIR)
print(f"Contest: {len(CONTEST_VIDEO_PATHS)} videos | {CONTEST_DIR}")


## 2. Browse and create a baseline

Inspect videos and record your guesses before computing metrics. The baseline predicts **real (0)** for every video; it covers the complete contest, regardless of the metric subset.

In [ ]:
video_browser = browse_contest(CONTEST_VIDEO_PATHS)
baseline_path = write_binary_submission(CONTEST_VIDEO_PATHS, WORK_DIR / "baseline_all_real.csv")
from IPython.display import FileLink, display
display(FileLink(str(baseline_path)))
# Download when ready:
# from google.colab import files
# files.download(str(baseline_path))

## 3. Set up selected metrics

Select T4 GPU for live metrics. Only the chosen evaluators and checkpoints are installed. Set `SUBJECT` to the visible object for PAS/TCS; OIS applies to people. These scores are diagnostics, not real/generated probabilities. MSS is excluded from this T4 workshop.

In [ ]:
VIDEO_PATH, VIDEO_PATHS = None, []
if RUN_METRICS:
    if not isinstance(MAX_METRIC_VIDEOS, int) or MAX_METRIC_VIDEOS < 1:
        raise ValueError("MAX_METRIC_VIDEOS must be a positive integer.")
    if not METRICS_TO_RUN or set(METRICS_TO_RUN) - {"PAS", "OIS", "TCS", "CAS"}:
        raise ValueError("Select PAS, OIS, TCS and/or CAS for this T4 notebook.")
    available = {p.stem: p for p in CONTEST_VIDEO_PATHS}
    selected_ids = METRIC_VIDEO_IDS or list(available)[:MAX_METRIC_VIDEOS]
    if len(set(selected_ids)) != len(selected_ids) or set(selected_ids) - set(available):
        raise ValueError("Select unique video IDs from the downloaded contest.")
    workshop = initialize(PROJECT_DIR, metrics=METRICS_TO_RUN,
                          create_variants=CREATE_VARIANTS, work_dir=WORK_DIR)
    workshop.input_paths = [available[video_id] for video_id in selected_ids]
    video_info_df = workshop.preview_videos()
    VIDEO_PATH, VIDEO_PATHS = workshop.select_videos()
else:
    print("Live metrics skipped. Videos and the all-real baseline are ready.")

## 4. Speed experiment (optional)

Enable `CREATE_VARIANTS` for fast ×2, slow ×0.5, frame drop and duplication. Variants are experiments, not extra contest entries.

In [ ]:
if RUN_METRICS and CREATE_VARIANTS:
    VARIANTS = workshop.prepare_variants()
    VIDEO_PATH, VIDEO_PATHS = workshop.select_videos()

## PAS

Subject displacement with camera-motion compensation. More motion does not establish authenticity.
`subject` is an English GroundingDINO prompt before SAM. Thresholds filter detections; `grid_size` controls tracker points.


In [ ]:
if RUN_METRICS and "PAS" in METRICS_TO_RUN:
    for path in VIDEO_PATHS:
        pas = compute_pas(path, subject=SUBJECT, box_threshold=0.25,
                          text_threshold=0.20, grid_size=30)
    if CREATE_VARIANTS:
        compare_variants("PAS", subject=SUBJECT)


## OIS

Human structural integrity. N/A when a person or pose cannot be detected.

In [ ]:
if RUN_METRICS and "OIS" in METRICS_TO_RUN:
    for path in VIDEO_PATHS:
        ois = compute_ois(path, bbox_thr=0.3, nms_thr=0.3)
    if CREATE_VARIANTS:
        compare_variants("OIS")


## TCS

Object persistence. Check detection warnings when interpreting high scores.

In [ ]:
if RUN_METRICS and "TCS" in METRICS_TO_RUN:
    for path in VIDEO_PATHS:
        tcs = compute_tcs(path, subject=SUBJECT, box_threshold=0.25,
                          text_threshold=0.20, grid_size=50)
    if CREATE_VARIANTS:
        compare_variants("TCS", subject=SUBJECT)


## CAS

Learned plausibility, not a physics simulator or a calibrated real/generated classifier.
Default evaluation uses 10 segments × 3 crops. Changing these changes sampling and can change the score.


In [ ]:
if RUN_METRICS and "CAS" in METRICS_TO_RUN:
    for path in VIDEO_PATHS:
        cas = compute_cas(path, test_num_segment=10, test_num_crop=3, num_workers=1)
    if CREATE_VARIANTS:
        compare_variants("CAS")


## Results

Collect the scores above without running evaluators again. Missing scores are not zeros; inspect status and runtime.

In [ ]:
if RUN_METRICS:
    workshop.show_metric_summary(VIDEO_PATH)
    results_df = workshop.evaluate_dataset(recompute=False)
    workshop.plot_results(results_df)

## Submit to Kaggle

Download `baseline_all_real.csv` and edit its `label` column, or fill `PREDICTIONS` below for every contest ID. Upload the CSV to Kaggle manually. Accuracy = correct labels / all videos; a perfect submission scores 1.0.

In [ ]:
PREDICTIONS = {}  # Example format: {"a1b2c3": 0, "d4e5f6": 1}; use actual contest IDs.
if PREDICTIONS:
    submission_path = write_binary_submission(CONTEST_VIDEO_PATHS, WORK_DIR / "submission.csv",
                                              predictions=PREDICTIONS)
    display(FileLink(str(submission_path)))
else:
    print("Edit the baseline CSV or fill PREDICTIONS for every contest video.")

## Discuss

- Did any metric change your initial guess?
- Where did a score disagree with your visual judgment?
- Why can a generated video receive high motion scores?